# Anima Caption (Colab)

Caption every image in `./source` with an Anima-style prompt.

**Runtime requirement:** GPU with enough VRAM for Gemma 4 26B-A4B at Q8 (A100 40GB / Colab Pro+). The Harrier embedding model (~270 MB) shares the same GPU.

1. Set the runtime to **GPU > A100**.
2. Run every cell top-to-bottom. The VLM, embedding model, and DB all download on first run (cached for subsequent runs).
3. Upload images into `./source/`, then run the captioning cell. Each image gets a sidecar `.txt`.
4. The last cell zips the captions for download.

## 1. Configuration

Edit the two repo IDs if you fork/rehost. Everything else has sensible defaults.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# --- Where to fetch artifacts from ----------------------------------------

# Hugging Face dataset that hosts the prebuilt danbooru.db (you upload this once;
# see README.md for the upload commands).
DB_REPO = "JackBinary/danbooru-db"          # <-- EDIT ME
DB_FILE = "danbooru.db"

# Hugging Face repo for the VLM GGUF. Defaults point at unsloth's public repo;
# change to your own mirror if you'd rather not depend on it.
VLM_REPO = "unsloth/gemma-4-26B-A4B-it-GGUF"
VLM_FILE = "gemma-4-26B-A4B-it-Q8_0.gguf"
VLM_MMPROJ = "mmproj-F16.gguf"                          # multimodal projector

# --- This GitHub repo (the one containing this notebook) ------------------
GITHUB_REPO = "JackBinary/caption-colab"     # <-- EDIT ME

# --- Where the images live ------------------------------------------------
# Folder of images to caption. Sidecar .txt files are written next to each
# image. Set to a Google Drive path (e.g. "/content/drive/MyDrive/captions/run1")
DATASET_DIR = "/content/drive/MyDrive/Loras/"

# --- Inference settings ---------------------------------------------------
# Single in-process llama-cpp-python instance; images are captioned serially.
N_CTX = 65536

## 2. Fetch this repo's code into Colab

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/content/caption-colab")
if not WORK.exists():
    subprocess.check_call(["git", "clone", "--depth", "1",
                           f"https://github.com/{GITHUB_REPO}.git", str(WORK)])
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("working dir:", os.getcwd())

## 3. Install dependencies

Installs `llama-cpp-python` from the **JamePeng fork** (which ships a `Gemma4ChatHandler` for multimodal Gemma 4) plus everything in `requirements.txt`. The fork publishes prebuilt CUDA wheels — no cmake build, no `llama-server`.

In [ ]:
import sys, subprocess

!pip install -q -r requirements.txt

PY_TAG = f"cp{sys.version_info.major}{sys.version_info.minor}"
WHEEL_URL = (
    "https://github.com/JamePeng/llama-cpp-python/releases/download/"
    "v0.3.39-cu128-linux-20260519/"
    f"llama_cpp_python-0.3.39%2Bcu128-{PY_TAG}-{PY_TAG}-linux_x86_64.whl"
)
print("installing:", WHEEL_URL)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", WHEEL_URL])

## 4. Download artifacts

- `danbooru.db` from your Hugging Face dataset
- VLM GGUF + mmproj from your HF model repo

In [ ]:
from huggingface_hub import hf_hub_download

DB_PATH = hf_hub_download(DB_REPO, DB_FILE, repo_type="dataset")
print("danbooru.db:", DB_PATH)

VLM_PATH = hf_hub_download(VLM_REPO, VLM_FILE)
MMPROJ_PATH = hf_hub_download(VLM_REPO, VLM_MMPROJ)
print("VLM:   ", VLM_PATH)
print("mmproj:", MMPROJ_PATH)

## 5. Load the VLM

Loads the GGUF + multimodal projector into a single `Llama` instance with `Gemma4ChatHandler`. Stays in memory until the runtime is restarted (or you run the shutdown cell).

In [ ]:
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Gemma4ChatHandler

chat_handler = Gemma4ChatHandler(clip_model_path=MMPROJ_PATH, verbose=False)
llm = Llama(
    model_path=VLM_PATH,
    chat_handler=chat_handler,
    n_gpu_layers=-1,
    n_ctx=N_CTX,
    verbose=False,
)
print("loaded VLM:", VLM_FILE)

## 6. Point at your dataset

`DATASET_DIR` from the config cell is where the captioner reads images and writes sidecar `.txt` files. Three common ways to populate it:

- **Google Drive** — set `DATASET_DIR = "/content/drive/MyDrive/..."` in the config cell, then run the mount cell below. Runs persist across Colab sessions.
- **File picker** — leave `DATASET_DIR = "./source"`, then uncomment the `files.upload()` block below.
- **`wget` / `gdown`** — fetch a zip into `DATASET_DIR` and unzip in place.

The captioner picks up every `.png` / `.jpg` / `.jpeg` / `.webp` / `.bmp` it finds there.

In [ ]:
from pathlib import Path
SOURCE = Path(DATASET_DIR)
SOURCE.mkdir(parents=True, exist_ok=True)

# Uncomment to grab files via the Colab uploader (writes into DATASET_DIR):
# from google.colab import files
# uploaded = files.upload()
# for name, data in uploaded.items():
#     (SOURCE / name).write_bytes(data)

imgs = sorted(p.name for p in SOURCE.iterdir()
              if p.suffix.lower() in {'.png','.jpg','.jpeg','.webp','.bmp'})
print(f"{len(imgs)} image(s) in {SOURCE}:")
for n in imgs[:20]:
    print(" ", n)
if len(imgs) > 20:
    print(f"  ... +{len(imgs)-20} more")

## 7. Caption the images

Runs the captioner over every image in `DATASET_DIR`. Each image gets a sidecar `.txt` next to it; re-running skips images that already have a caption (`overwrite=True` to force).

In [ ]:
from caption import caption_all

done, errors = caption_all(
    source_dir=DATASET_DIR,
    llm=llm,
    db_path=DB_PATH,
    anima_md_path="./Anima_prompting.md",
    overwrite=False,
)